## Imports and paths

In [1]:
from config import *
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from itertools import combinations
from sklearn.metrics import cohen_kappa_score, accuracy_score
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa
pd.set_option("display.max_columns", None)  # show all cols
pd.set_option("display.width", None)        # + better readability

In [2]:
# shared plot config, same conventions as streamlit_app.py
import plotly.io as pio

# default plotly look, but with no background, and a black left + bottom axis line with outside ticks
thesis_template = go.layout.Template(pio.templates["plotly"])
thesis_template.layout.update(
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="white",
    font=dict(size=15),   # plotly default is 12, too small in the report; titles, ticks, legend and bar text scale from this
)

axis_style = dict(
    showline=True,
    linecolor="black",
    linewidth=1,
    ticks="outside",
    tickcolor="black",
    ticklen=5,
    showgrid=False,
    zeroline=False,
    mirror=False,   # only left + bottom, no box around the plot
)
thesis_template.layout.xaxis.update(axis_style)
thesis_template.layout.yaxis.update(axis_style)
pio.templates["thesis"] = thesis_template
pio.templates.default = "thesis"

# one fixed color per model, so a model looks the same in every plot regardless of sort order
MODEL_NAMES = ['claude-sonnet-5', 'gemini-3.7-flash', 'gpt-5.6-luna', 'gpt-5.6-terra']
MODEL_COLORS = dict(zip(MODEL_NAMES, px.colors.qualitative.Plotly))

SEQUENTIAL_SCALE = px.colors.sequential.Viridis
ANNOTATION_FONT = dict(size=14, color="#555555", family="monospace")
MARGIN = {"r": 20, "t": 80, "l": 20, "b": 60}

def plot_title(title, subtitle=None):
    # bold title + optional small subtitle line, e.g. "n = 439 restaurants"
    return f"<b>{title}</b>" + (f"<br><sup>{subtitle}</sup>" if subtitle else "")

def save_plot(fig, filename):
    fig.write_image(PLOTS_DIR / filename)     # comment in if needed to save plots
    return

In [3]:
csv_path = API_DATA_DIR / 'responses'
raw_path = API_DATA_DIR / 'raw_responses'

## Model Cost and Token Usage

### Model Cost in USD 

In [4]:
files = sorted(raw_path.glob('20260917-*'))

dfs = []

for f in files:
    df_raw = pd.read_json(f, lines=True)[['model', 'batch_index', 'usage']]
    dfs.append(df_raw)

raw_df = pd.concat(dfs, ignore_index=True)
# unpack usage column
usage_df = pd.json_normalize(raw_df['usage'])
raw_df = pd.concat([raw_df.drop(columns='usage'), usage_df], axis=1)

raw_df.head()

,model,batch_index,completion_tokens,prompt_tokens,total_tokens,reasoning_tokens,cost,completion_tokens_details.accepted_prediction_tokens,completion_tokens_details.audio_tokens,completion_tokens_details.reasoning_tokens,completion_tokens_details.rejected_prediction_tokens,completion_tokens_details.text_tokens,completion_tokens_details.image_tokens,prompt_tokens_details.audio_tokens,prompt_tokens_details.cache_write_tokens,prompt_tokens_details.cached_tokens,prompt_tokens_details.image_tokens,prompt_tokens_details.text_tokens,prompt_tokens_details.video_tokens,cost_details.upstream_inference_cost,cost_details.upstream_inference_prompt_cost,cost_details.upstream_inference_completions_cost,cost_details.total_cost,cost_details.input_cost,cost_details.output_cost,cost_details.cached_input_cost,cost_details.cache_write_input_cost,cost_details.request_cost,cost_details.web_search_cost,cost_details.image_input_cost,cost_details.image_output_cost,cost_details.audio_input_cost,prompt_tokens_details.cache_creation_tokens
0,anthropic/claude-sonnet-5,0,6787,422164,428951,3342,1.132198,None,0,3342,None,None,0,0,0,0,0,None,0,0.912198,0.844328,0.06787,1.132198,0.844328,0.06787,0.0,0.0,0,0.22,None,None,None,NaN
1,anthropic/claude-sonnet-5,1,5477,528754,534231,1690,1.342278,None,0,1690,None,None,0,0,0,0,0,None,0,1.112278,1.057508,0.05477,1.342278,1.057508,0.05477,0.0,0.0,0,0.23,None,None,None,NaN
2,anthropic/claude-sonnet-5,2,9156,689256,698412,5519,1.710072,None,0,5519,None,None,0,0,0,0,0,None,0,1.470072,1.378512,0.09156,1.710072,1.378512,0.09156,0.0,0.0,0,0.24,None,None,None,NaN
3,anthropic/claude-sonnet-5,3,7346,500708,508054,3612,1.314876,None,0,3612,None,None,0,0,0,0,0,None,0,1.074876,1.001416,0.07346,1.314876,1.001416,0.07346,0.0,0.0,0,0.24,None,None,None,NaN
4,anthropic/claude-sonnet-5,4,9081,711779,720860,4953,1.804368,None,0,4953,None,None,0,0,0,0,0,None,0,1.514368,1.423558,0.09081,1.804368,1.423558,0.09081,0.0,0.0,0,0.29,None,None,None,NaN


In [5]:

# # including sonnet cached for one batch
# sonnet_cache = pd.read_json(raw_path / '20260919-124908-claude-sonnet-5.jsonl', lines=True)[['model', 'batch_index', 'usage']]
# sonnet_usage_df = pd.json_normalize(sonnet_cache['usage'])
# sonnet_cache = pd.concat([sonnet_cache.drop(columns='usage'), sonnet_usage_df], axis=1)

# sonnet_cache['model'] = sonnet_cache['model'].replace({ 'anthropic/claude-sonnet-5': 'anthropic/claude-sonnet-5-cached' }) # keep cached as own entry
# raw_df = pd.concat([sonnet_cache, raw_df], axis=0)
# raw_df

In [6]:
# gemini ran through two providers (google-vertex + google-ai-studio) -> strip the provider prefix so they're grouped together
raw_df['model_name'] = raw_df['model'].str.split('/').str[1]

# one row per batch call -> sum the cost of all batches per model
model_price = raw_df.groupby('model_name').agg(n_batches=('batch_index', 'nunique'), total_cost=('cost', 'sum'))
model_price.head()

,n_batches,total_cost
model_name,,
claude-sonnet-5,22,29.838944
gemini-3.7-flash,22,3.862064
gpt-5.6-luna,22,2.832598
gpt-5.6-terra,22,8.526132


In [7]:
plot_df = model_price.reset_index()

fig = px.bar(
    plot_df,
    x="total_cost",
    y="model_name",
    color="model_name",
    orientation="h",
    title=plot_title("Total cost per model", f"10% sample · n = 439"),
    labels={
        "total_cost": "Total cost (USD)",
        "model_name": "Model",
    },
)

fig.update_traces(
    width=0.6,
    texttemplate="$%{x:.2f}",
    textposition="outside",
    hovertemplate=None,
    hoverinfo="skip"
)

fig.update_layout(
    xaxis=dict(range=[0, plot_df["total_cost"].max() * 1.15]),
    yaxis=dict(categoryorder="total descending"),
    height=350,
    showlegend=False,
    margin=MARGIN,
)

fig.update_xaxes(
    showgrid=True, 
    gridcolor="#e5e5e5", 
    gridwidth=0.5
)

save_plot(fig, 'total-cost-per-model-10%.svg')
fig.show()

### Model Tokens

In [8]:
model_tokens = raw_df.groupby('model_name').agg(n_batches=('batch_index', 'nunique'), total_tokens=('total_tokens', 'sum'))
model_tokens.head()

,n_batches,total_tokens
model_name,,
claude-sonnet-5,22,11802600
gemini-3.7-flash,22,264219
gpt-5.6-luna,22,1944404
gpt-5.6-terra,22,2393997


In [9]:
plot_df = model_tokens.reset_index()
# show in millions: keeps the linear scale (bar length = real ratio) but readable numbers
plot_df['total_tokens_m'] = plot_df['total_tokens'] / 1e6

fig = px.bar(
    plot_df,
    x="total_tokens_m",
    y="model_name",
    color="model_name",
    color_discrete_map=MODEL_COLORS,
    orientation="h",
    title=plot_title("Total tokens per model", f"10% sample · n = 439"),
    labels={
        "total_tokens_m": "Total tokens (millions)",
        "model_name": "Model",
    },
)

fig.update_traces(
    width=0.6,
    texttemplate="%{x:.2f}M",
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Total tokens: %{x:.2f}M<extra></extra>",
)

fig.update_layout(
    xaxis=dict(range=[0, plot_df["total_tokens_m"].max() * 1.15]),
    # smallest bar on top
    yaxis=dict(categoryorder="total descending"),
    height=350,
    showlegend=False,
    margin=MARGIN,
)

fig.update_xaxes(
    showgrid=True, 
    gridcolor="#e5e5e5", 
    gridwidth=0.5
)

save_plot(fig, 'total-tokens-pr-model-10%.svg')
fig.show()

## Load model answers

One row per restaurant and model.

`hasFoundMenu` is always consistent with `menuLink`, so we use `hasFoundMenu` as the binary label.

In [10]:
files = sorted(csv_path.glob('20260917-*'))

dfs = []
for f in files:
    df = pd.read_csv(f, usecols=['model', 'input', 'answer'])
    answer = pd.json_normalize(df['answer'].apply(json.loads))
    # websiteUri is only in the input we sent to the model
    answer['websiteUri'] = df['input'].apply(lambda s: json.loads(s).get('websiteUri'))
    answer['model'] = df['model'].str.split('/').str[1]
    dfs.append(answer)

df = pd.concat(dfs, ignore_index=True)
df.head()


,id,name,hasFoundMenu,menuLink,format,source,websiteUri,model
0,ChIJAV1_vX1SUkYRifHQGs0-33o,Pizze Di Napo Hellerup,True,https://pizzedinapohellerup.dk/,HTML,OFFICIAL_WEBSITE,https://platia.dk/pizze-di-napo-hellerup,claude-sonnet-5
1,ChIJ7RIaXXpQUkYRxGD3xBRRltk,McDonald's,True,https://www.mcdonalds.com/dk/da-dk/vores-menu....,HTML,OFFICIAL_WEBSITE,https://www.mcdonalds.com/dk/da-dk/location/ba...,claude-sonnet-5
2,ChIJq6p-9w9TUkYRGJ7PByHSDYY,Byens Pizza,True,http://byens-pizza.dk/?page_id=2065,HTML,OFFICIAL_WEBSITE,http://byens-pizza.dk/?page_id=2065,claude-sonnet-5
3,ChIJz3lHpeRSUkYRRNH0_-MSMhU,Restaurant Taormina v/Mukuntharaj Karunaratnam,True,https://taormina.dk/menu/,HTML,OFFICIAL_WEBSITE,https://taormina.dk/,claude-sonnet-5
4,ChIJBVGBtQFTUkYR7R1FSqjW1kg,Barnetsfest,False,,,,http://www.barnetsfest.dk/,claude-sonnet-5


### Restaurant IDs and menu found

Pivot the table, so that we get the dataframe by whether or not a menu was found, with each model as its own column.

In [11]:
found = df.pivot_table(index='id', columns='model', values='hasFoundMenu', aggfunc='first').astype(bool)
models = list(found.columns)

print(f"{len(found)} restaurants, {len(models)} models: {models}")
found.head()

439 restaurants, 4 models: ['claude-sonnet-5', 'gemini-3.7-flash', 'gpt-5.6-luna', 'gpt-5.6-terra']


model,claude-sonnet-5,gemini-3.7-flash,gpt-5.6-luna,gpt-5.6-terra
id,,,,
ChIJ-7iMbc1TUkYR066Hz1SQ-EI,True,True,True,True
ChIJ-QrW55VTUkYRiO5rYIpA5PE,True,True,True,True
ChIJ-XF3Wu1TUkYRl25fQ2IL8Ps,True,True,True,True
ChIJ-YewRThSUkYRU-ZX1KiCAr0,True,True,True,True
ChIJ-ao52_ZNUkYRU11OTne3ec8,True,True,True,True


### How many models said "found"?

For each restaurant, we want to figure out for how many restaurants, all four models agreed. This gives us an overall view of how many entries were ambigous.

- 0 and 4 means that either *no* model found a menu or *all* models found a menu. This gives us a somewhat confidence in that they agree that this restaurant had a menu.
- On the other hand, 1, 2, 3 are ambigous, since we don't have a ground truth.

In [12]:
votes = found.sum(axis=1)
vote_counts = votes.value_counts().sort_index()
full_agreement = vote_counts.get(0, 0) + vote_counts.get(len(models), 0)

plot_df = vote_counts.rename_axis('n_found').reset_index(name='restaurants')
plot_df['agreement'] = np.where(plot_df['n_found'].isin([0, len(models)]), 'Full agreement', 'Disagreement')

fig = px.bar(
    plot_df,
    x="n_found",
    y="restaurants",
    color="agreement",
    # unanimous (0 or 4) in a lighter blue, Disagreement in the darker one
    color_discrete_map={"Full agreement": "#B1B7FD", "Disagreement": "#636EFA"},
    title=plot_title(
        "Model votes per restaurant on hasFoundMenu",
        f"n = {len(found):,} restaurants · {full_agreement} full agreement ({full_agreement / len(found):.1%})",
    ),
    labels={
        "n_found": "Number of models that found a menu link",
        "restaurants": "Restaurants",
        "agreement": "",
    },
)

fig.update_traces(
    width=0.4,
    texttemplate="%{y}",
    textposition="outside",
    hovertemplate="<b>%{x} of 4 models</b> found a menu<br>Restaurants: %{y}<extra></extra>",
)

fig.update_layout(
    xaxis=dict(type="category", categoryorder="array", categoryarray=list(range(len(models) + 1))),
    yaxis=dict(range=[0, plot_df["restaurants"].max() * 1.15]),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
    ),
    # bargap=0.4,
    margin=MARGIN,
)

fig.update_yaxes(
    showgrid=True, 
    gridcolor="#e5e5e5", 
    gridwidth=0.5
)

save_plot(fig, 'hasfoundmenu-votes-per-restaurant-10%.svg')
fig.show()

print(f"Full agreement: {full_agreement}/{len(found)} ({full_agreement / len(found):.1%})")

Full agreement: 274/439 (62.4%)


This leavus us with **165** restaurant IDs that are ambigous, which we would have to manually check.

## Pairwise agreement

- **Percent agreement**: share of restaurants where both models give the same answer.
- **Cohen's kappa**: same, but corrected for the agreement you'd expect by chance given how often each model says "found". Since most restaurants have a menu, two models that both say "found" a lot will agree often by pure chance, so kappa is the fairer number.

Rough reading of kappa (Landis & Koch): < 0.2 slight, 0.2-0.4 fair, 0.4-0.6 moderate, 0.6-0.8 substantial, > 0.8 almost perfect.

In [13]:
pct_agreement = pd.DataFrame(1.0, index=models, columns=models)
cohen = pd.DataFrame(1.0, index=models, columns=models)

for model_1, model_2 in combinations(models, 2):
    pct = accuracy_score(found[model_1], found[model_2])
    k = cohen_kappa_score(found[model_1], found[model_2])
    pct_agreement.loc[model_1, model_2] = pct_agreement.loc[model_2, model_1] = pct
    cohen.loc[model_1, model_2] = cohen.loc[model_2, model_1] = k

pd.concat({'percent agreement': pct_agreement, "cohen's kappa": cohen}, axis=1).round(3)

percent agreement                                \
                   claude-sonnet-5 gemini-3.7-flash gpt-5.6-luna   
claude-sonnet-5              1.000            0.763        0.749   
gemini-3.7-flash             0.763            1.000        0.809   
gpt-5.6-luna                 0.749            0.809        1.000   
gpt-5.6-terra                0.747            0.861        0.834   

                                 cohen's kappa                                \
                 gpt-5.6-terra claude-sonnet-5 gemini-3.7-flash gpt-5.6-luna   
claude-sonnet-5          0.747           1.000            0.491        0.478   
gemini-3.7-flash         0.861           0.491            1.000        0.558   
gpt-5.6-luna             0.834           0.478            0.558        1.000   
gpt-5.6-terra            1.000           0.463            0.653        0.624   

                                
                 gpt-5.6-terra  
claude-sonnet-5          0.463  
gemini-3.7-flash         0.653  
gpt-5.6-luna             0.624  
gpt-5.6-terra            1.000

In [14]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["Percentage", "Cohen's kappa"], horizontal_spacing=0.1, vertical_spacing=0.1)

for col, matrix in enumerate([pct_agreement, cohen], start=1):
    # upper triangle is a mirror of the lower one and the diagonal is always 1 -> drop the first row
    # and last column (they would only hold the diagonal) and blank out the rest of the upper triangle
    lower = matrix.iloc[1:, :-1]
    z = lower.where(np.tril(np.ones(lower.shape, dtype=bool)))

    fig.add_trace(go.Heatmap(
        z=z.values,
        x=lower.columns,
        y=lower.index,
        coloraxis="coloraxis",
        texttemplate="%{z:.2f}",
        xgap=2.5,
        ygap=2.5,
    ), row=1, col=col)

fig.update_layout(
    title=plot_title("Inter Annotator Agreement on hasFoundMenu", f"10% sample · n = {len(found):,} restaurants"),
    coloraxis=dict(
        colorscale=SEQUENTIAL_SCALE,
        cmin=0,
        cmax=1,
        colorbar_title="Agreement",
    ),
    plot_bgcolor="white",
    height=500,
    width=1000,
    margin=MARGIN,
)
fig.update_xaxes(constrain="domain")
fig.update_yaxes(constrain="domain")

# top row first, like a normal matrix, and square cells
fig.update_yaxes(autorange="reversed", scaleanchor="x", automargin=True)
fig.update_yaxes(scaleanchor="x2", row=1, col=2)
fig.update_xaxes(tickangle=-90, automargin=True)
save_plot(fig, 'hasfoundmenu-pairwise-agreement-10%.svg')
fig.show()

## Fleiss' kappa - all 4 models at once

In [15]:
def fleiss(matrix):
    table, _ = aggregate_raters(matrix)
    return fleiss_kappa(table)

kappa_all = fleiss(found.to_numpy().astype(int))

print(f"Fleiss' kappa (hasFoundMenu, {len(models)} models, n={len(found)}): {kappa_all:.3f}")

Fleiss' kappa (hasFoundMenu, 4 models, n=439): 0.538


## Annotation Analyser

In [16]:
annotations = pd.read_csv(API_DATA_DIR / '10%_menulink_annotations.csv')
annotations.head()

,id,model,url,category,notes
0,ChIJ-7iMbc1TUkYR066Hz1SQ-EI,claude-sonnet-5,https://www.restaurantalf.dk/,correct_website,NaN
1,ChIJ-7iMbc1TUkYR066Hz1SQ-EI,gemini-3.7-flash,https://restaurantalf.dk/,correct_website,NaN
2,ChIJ-7iMbc1TUkYR066Hz1SQ-EI,gpt-5.6-luna,https://www.restaurantalf.dk/,correct_website,NaN
3,ChIJ-7iMbc1TUkYR066Hz1SQ-EI,gpt-5.6-terra,https://www.restaurantalf.dk/,correct_website,NaN
4,ChIJ-QrW55VTUkYRiO5rYIpA5PE,claude-sonnet-5,https://durumbar-frb.dk/,broken_link,NaN


In [17]:
category_order = ["correct_website", "correct_thirdparty", "broken_link", "website_not_menu", "incorrect", "other",]

category_counts_raw = (
    annotations.groupby('model')['category']
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=category_order)
)

category_counts_raw

category,correct_website,correct_thirdparty,broken_link,website_not_menu,incorrect,other
model,,,,,,
claude-sonnet-5,40,13,12,8,4,2
gemini-3.7-flash,30,3,25,41,0,1
gpt-5.6-luna,40,15,7,14,4,6
gpt-5.6-terra,58,17,6,5,4,4


In [18]:
# plotly's default blue + red with lighter tints (same as the votes plot) -> blue = correct, red = incorrect, grey = other
CATEGORY_COLORS = {
    'correct_website': '#636EFA',
    'correct_thirdparty': '#B1B7FD',
    'broken_link': '#EF553B',
    'website_not_menu': '#F59989',
    'incorrect': '#FBD4CE',
    'other': '#D3D3D3',
}

In [19]:
correct_categories = ['correct_website', 'correct_thirdparty']
incorrect_categories = ['broken_link', 'website_not_menu', 'incorrect', 'other']
n_per_model = category_counts_raw.sum(axis=1)
# percentage of each model's own n, since the models don't have the same number of annotated links
category_share = category_counts_raw.div(n_per_model, axis=0) * 100
pct_correct = category_share[correct_categories].sum(axis=1)
model_labels = [f"{m} (n = {n_per_model[m]})" for m in category_counts_raw.index]

# ascending, since the first category is drawn at the bottom -> highest share of correct responses on top
model_order = [f"{m} (n = {n_per_model[m]})" for m in pct_correct.sort_values().index]

fig = go.Figure()

for group, categories in [("Correct", correct_categories), ("Incorrect", incorrect_categories)]:
    for category in categories:
        pct = category_share[category].values
        fig.add_trace(go.Bar(
            x=pct,
            y=model_labels,
            orientation="h",
            name=category,
            marker_color=CATEGORY_COLORS[category],
            legendgroup=group,
            legendgrouptitle_text=group,
            customdata=np.stack([pct, category_counts_raw[category].values], axis=-1),
            text=[f"{v:.0f}%" if v > 0 else "" for v in pct],
            textposition="inside",
            insidetextanchor="middle",
            textangle=0,
            hovertemplate=f"<b>%{{y}}</b><br>{category}: %{{customdata[0]:.1f}}% (%{{customdata[1]}})<extra></extra>",
        ))

fig.update_traces(width=0.6, marker_line_color="white", marker_line_width=1)

fig.update_layout(
    title=plot_title("Correctness of validated menu links by model", "10% sample · manually validated · share of each model's n"),
    barmode="stack",
    xaxis=dict(title="Percentage (%)", range=[0, 100], dtick=25),
    yaxis=dict(title="Model", categoryorder="array", categoryarray=model_order),
    uniformtext=dict(minsize=8, mode="hide"),
    height=450,
    width=1000,
    legend=dict(
        orientation="v",
        xanchor="left",
        traceorder="grouped",  # plotly reverses the legend for stacked bars by default
    ),
    margin=MARGIN,
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="#e5e5e5",
    gridwidth=0.5
)

save_plot(fig, 'correctes-validated-menu-links-pr-model-10%.svg')
fig.show()